# Part-of-Speech Tagging with Convolutional Neural Networks

In this notebook, we'll implement a Part-of-Speech (POS) tagger using Convolutional Neural Networks (CNNs). We'll use a stacked CNN with n-gram filters (n = 2, 3, 4), residual connections, and a dense layer with softmax at the top layer.


In [ ]:
# !pip uninstall -y numpy scipy tensorflow tensorflow-gpu
# !pip install numpy==1.23.5
# !pip install scipy==1.10.1
# !pip install tensorflow==2.12.0
# !pip install gensim==4.3.1
# !pip install conllu

In [ ]:
import tensorflow as tf

# Enable memory growth for GPUs (optional but safer)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is available and memory growth is set.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Running on CPU.")


## 1. Libraries Import


In [ ]:
import os
import urllib.request
import zipfile
import conllu
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Embedding, Conv1D, Dense, Dropout,
                                   TimeDistributed, Flatten, Layer, GlobalMaxPooling1D,
                                   Concatenate, Add, Masking, Lambda)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from collections import Counter, defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc
import matplotlib.pyplot as plt
import gensim.downloader as api
from tensorflow.keras.regularizers import l2

from sklearn.metrics import confusion_matrix
import seaborn as sns
from tensorflow.keras.utils import register_keras_serializable


## 2. Data Loading and Preprocessing


In [ ]:
# Constants
LANGUAGE = "English"
TREEBANK = "UD_English-EWT"
TREEBANK_URL = "https://github.com/UniversalDependencies/UD_English-EWT/archive/master.zip"
MAX_SEQUENCE_LENGTH = 50  # Maximum sentence length
EMBEDDING_DIM = 100  # Dimension of word embeddings
CHAR_EMBEDDING_DIM = 50  # Dimension of character embeddings
MAX_WORD_LENGTH = 20  # Maximum word length for character-level features
BATCH_SIZE = 32
EPOCHS = 50
WINDOW_SIZE = 2
CNN_FILTERS = 128  # Number of filters for CNN
NUM_CNN_LAYERS = 2  # Number of stacked CNN layers
DROPOUT_RATE = 0.3
USE_CHAR_EMBEDDINGS = True  # Whether to use character-level embeddings


In [ ]:
# Function to download and extract the treebank
def download_treebank(url, treebank_name):
    zip_path = f"{treebank_name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {treebank_name}...")
        urllib.request.urlretrieve(url, zip_path)

    extract_dir = f"{treebank_name}-data"
    if not os.path.exists(extract_dir):
        print(f"Extracting {treebank_name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

    return extract_dir

# Function to load and parse the CoNLL-U files
def load_conllu_data(treebank_dir, treebank_name):
    base_dir = os.path.join(treebank_dir, f"{treebank_name}-master")
    train_file = None
    dev_file = None
    test_file = None

    for file in os.listdir(base_dir):
        if file.endswith(".conllu"):
            if "train" in file:
                train_file = os.path.join(base_dir, file)
            elif "dev" in file:
                dev_file = os.path.join(base_dir, file)
            elif "test" in file:
                test_file = os.path.join(base_dir, file)

    train_data = []
    dev_data = []
    test_data = []

    if train_file:
        with open(train_file, "r", encoding="utf-8") as f:
            train_data = conllu.parse(f.read())

    if dev_file:
        with open(dev_file, "r", encoding="utf-8") as f:
            dev_data = conllu.parse(f.read())

    if test_file:
        with open(test_file, "r", encoding="utf-8") as f:
            test_data = conllu.parse(f.read())

    return train_data, dev_data, test_data

# Function to extract sentences and POS tags from the parsed data
def extract_sentences_and_tags(data):
    sentences = []
    pos_tags = []

    for sentence in data:
        words = []
        tags = []

        for token in sentence:
            if token["upos"] != "_":
                words.append(token["form"].lower())
                tags.append(token["upos"])

        if words:
            sentences.append(words)
            pos_tags.append(tags)

    return sentences, pos_tags


In [ ]:
# Download and load data
print("Loading data...")
treebank_dir = download_treebank(TREEBANK_URL, TREEBANK)
train_data, dev_data, test_data = load_conllu_data(treebank_dir, TREEBANK)

# Extract sentences and tags
train_sentences, train_pos_tags = extract_sentences_and_tags(train_data)
dev_sentences, dev_pos_tags = extract_sentences_and_tags(dev_data)
test_sentences, test_pos_tags = extract_sentences_and_tags(test_data)


In [ ]:
# Calculate dataset statistics
def calculate_dataset_stats(sentences, tags):
    num_sentences = len(sentences)
    num_words = sum(len(s) for s in sentences)
    avg_sentence_length = num_words / num_sentences if num_sentences > 0 else 0

    # Calculate vocabulary size
    vocab = set()
    for sentence in sentences:
        vocab.update(sentence)
    vocab_size = len(vocab)

    # Calculate tag distribution
    tag_counter = Counter()
    for tag_seq in tags:
        tag_counter.update(tag_seq)

    return {
        "num_sentences": num_sentences,
        "num_words": num_words,
        "avg_sentence_length": avg_sentence_length,
        "vocab_size": vocab_size,
        "tag_distribution": tag_counter
    }

# Calculate and display dataset statistics
print("Calculating dataset statistics...")
train_stats = calculate_dataset_stats(train_sentences, train_pos_tags)
dev_stats = calculate_dataset_stats(dev_sentences, dev_pos_tags)
test_stats = calculate_dataset_stats(test_sentences, test_pos_tags)

print(f"Dataset Statistics for {LANGUAGE} ({TREEBANK}):\n")
print("Training Set:")
print(f"  Number of sentences: {train_stats['num_sentences']}")
print(f"  Number of words: {train_stats['num_words']}")
print(f"  Average sentence length: {train_stats['avg_sentence_length']:.2f}")
print(f"  Vocabulary size: {train_stats['vocab_size']}")
print("\nDevelopment Set:")
print(f"  Number of sentences: {dev_stats['num_sentences']}")
print(f"  Number of words: {dev_stats['num_words']}")
print(f"  Average sentence length: {dev_stats['avg_sentence_length']:.2f}")
print("\nTest Set:")
print(f"  Number of sentences: {test_stats['num_sentences']}")
print(f"  Number of words: {test_stats['num_words']}")
print(f"  Average sentence length: {test_stats['avg_sentence_length']:.2f}")

# Display tag distribution
print("\nPOS Tag Distribution (Training Set):")
for tag, count in sorted(train_stats['tag_distribution'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {tag}: {count} ({count/train_stats['num_words']*100:.2f}%)")


## 3. Word and Character Embeddings


In [ ]:
# Create vocabulary and tag mappings
def create_mappings(train_sentences, train_tags):
    # Create word-to-index mapping
    word_to_idx = {"<PAD>": 0, "<UNK>": 1}  # Special tokens
    for sentence in train_sentences:
        for word in sentence:
            if word not in word_to_idx:
                word_to_idx[word] = len(word_to_idx)

    # Create tag-to-index mapping
    tag_to_idx = {}
    for tag_seq in train_tags:
        for tag in tag_seq:
            if tag not in tag_to_idx:
                tag_to_idx[tag] = len(tag_to_idx)

    # Create index-to-tag mapping for later use
    idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}

    return word_to_idx, tag_to_idx, idx_to_tag

# Create character mappings
def create_char_mappings(sentences):
    char_to_idx = {"<PAD>": 0, "<UNK>": 1}
    for sentence in sentences:
        for word in sentence:
            for char in word:
                if char not in char_to_idx:
                    char_to_idx[char] = len(char_to_idx)

    return char_to_idx

# Load pre-trained word embeddings
print("\nLoading pre-trained word embeddings...")
word_vectors = api.load("glove-wiki-gigaword-100")  # 100-dimensional GloVe embeddings
EMBEDDING_DIM = word_vectors.vector_size
print(f"Loaded {len(word_vectors.key_to_index)} word vectors with dimension {EMBEDDING_DIM}")

# Create mappings
print("Creating word and tag mappings...")
word_to_idx, tag_to_idx, idx_to_tag = create_mappings(train_sentences, train_pos_tags)
print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Number of POS tags: {len(tag_to_idx)}")
print(f"POS tags: {list(tag_to_idx.keys())}")

if USE_CHAR_EMBEDDINGS:
    char_to_idx = create_char_mappings(train_sentences)
    print(f"Character vocabulary size: {len(char_to_idx)}")

# Create embedding matrix
def create_embedding_matrix(word_to_idx, word_vectors, embedding_dim):
    embedding_matrix = np.zeros((len(word_to_idx), embedding_dim))
    for word, idx in word_to_idx.items():
        if word in word_vectors:
            embedding_matrix[idx] = word_vectors[word]
        elif word == "<PAD>":
            embedding_matrix[idx] = np.zeros(embedding_dim)  # Zero vector for padding
        else:  # <UNK> or words not in pre-trained embeddings
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

    return embedding_matrix

print("Creating embedding matrix...")
embedding_matrix = create_embedding_matrix(word_to_idx, word_vectors, EMBEDDING_DIM)
print(f"Embedding matrix shape: {embedding_matrix.shape}")


In [ ]:
# Prepare character-level data
def prepare_char_data(sentences, char_to_idx, max_seq_length, max_word_length):
    X_char = []

    for sentence in sentences:
        sent_chars = []
        for word in sentence:
            word_chars = [char_to_idx.get(char, char_to_idx["<UNK>"]) for char in word[:max_word_length]]
            # Pad word to max_word_length
            word_chars = word_chars + [char_to_idx["<PAD>"]] * (max_word_length - len(word_chars))
            sent_chars.append(word_chars)

        # Pad sentence to max_seq_length
        if len(sent_chars) < max_seq_length:
            padding = [[char_to_idx["<PAD>"]] * max_word_length] * (max_seq_length - len(sent_chars))
            sent_chars = sent_chars + padding
        else:
            sent_chars = sent_chars[:max_seq_length]

        X_char.append(sent_chars)

    return np.array(X_char, dtype='int32')

# Prepare data for CNN
def prepare_data_for_cnn(sentences, pos_tags, word_to_idx, tag_to_idx, max_length):
    X = []
    y = []
    lengths = []  # Store original sequence lengths

    for sentence, tags in zip(sentences, pos_tags):
        # Convert words to indices
        word_indices = [word_to_idx.get(word, word_to_idx["<UNK>"]) for word in sentence]
        tag_indices = [tag_to_idx[tag] for tag in tags]

        # Store original length
        lengths.append(min(len(sentence), max_length))

        X.append(word_indices)
        y.append(tag_indices)

    # Pad sequences
    X_padded = pad_sequences(X, maxlen=max_length, padding='post', value=word_to_idx["<PAD>"])
    y_padded = pad_sequences(y, maxlen=max_length, padding='post', value=0)  # 0 is padding index

    return X_padded, y_padded, lengths


In [ ]:
# Prepare data
print("Preparing data...")
X_train, y_train, train_lengths = prepare_data_for_cnn(train_sentences, train_pos_tags,
                                        word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)
X_dev, y_dev, dev_lengths = prepare_data_for_cnn(dev_sentences, dev_pos_tags,
                                    word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)
X_test, y_test, test_lengths = prepare_data_for_cnn(test_sentences, test_pos_tags,
                                      word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)

train_inputs = [X_train]
dev_inputs = [X_dev]
test_inputs = [X_test]

if USE_CHAR_EMBEDDINGS:
    X_train_char = prepare_char_data(train_sentences, char_to_idx,
                                    MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)
    X_dev_char = prepare_char_data(dev_sentences, char_to_idx,
                                  MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)
    X_test_char = prepare_char_data(test_sentences, char_to_idx,
                                    MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)
    train_inputs.append(X_train_char)
    dev_inputs.append(X_dev_char)
    test_inputs.append(X_test_char)

print(f"Training samples: {len(X_train)}")
print(f"Max sequence length: {MAX_SEQUENCE_LENGTH}")


## 4. Baseline Tagger


In [ ]:
# Baseline Tagger
class BaselineTagger:
    def __init__(self):
        self.word_to_tag = {}
        self.most_common_tag = None

    def train(self, sentences, tags):
        word_tag_counts = defaultdict(Counter)
        tag_counts = Counter()

        for sentence, tag_seq in zip(sentences, tags):
            for word, tag in zip(sentence, tag_seq):
                word_tag_counts[word][tag] += 1
                tag_counts[tag] += 1

        for word, tag_counter in word_tag_counts.items():
            self.word_to_tag[word] = tag_counter.most_common(1)[0][0]

        self.most_common_tag = tag_counts.most_common(1)[0][0]

    def predict(self, sentences):
        predictions = []
        for sentence in sentences:
            sentence_preds = []
            for word in sentence:
                tag = self.word_to_tag.get(word, self.most_common_tag)
                sentence_preds.append(tag)
            predictions.append(sentence_preds)
        return predictions

# Train baseline model
print("Training baseline model...")
baseline = BaselineTagger()
baseline.train(train_sentences, train_pos_tags)

# Convert baseline predictions to sequence format
def convert_baseline_to_sequence_format(sentences, predictions, tag_to_idx, max_length):
    y_baseline = []
    for sent, pred in zip(sentences, predictions):
        sent_indices = [tag_to_idx[tag] for tag in pred]
        # Pad to max_length
        sent_indices = sent_indices + [0] * (max_length - len(sent_indices))
        y_baseline.append(sent_indices[:max_length])
    return np.array(y_baseline)

# Get baseline predictions
train_baseline_preds = baseline.predict(train_sentences)
dev_baseline_preds = baseline.predict(dev_sentences)
test_baseline_preds = baseline.predict(test_sentences)

# Convert to sequence format
train_baseline_seq = convert_baseline_to_sequence_format(train_sentences, train_baseline_preds,
                                                        tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_baseline_seq = convert_baseline_to_sequence_format(dev_sentences, dev_baseline_preds,
                                                      tag_to_idx, MAX_SEQUENCE_LENGTH)
test_baseline_seq = convert_baseline_to_sequence_format(test_sentences, test_baseline_preds,
                                                      tag_to_idx, MAX_SEQUENCE_LENGTH)


## 5. CNN Model

Now we'll implement a stacked CNN with n-gram filters (n = 2, 3, 4), residual connections, and a dense layer with softmax at the top layer for POS tagging.


In [ ]:
# Custom layers for masking
@register_keras_serializable()
class MaskNonPadding(Layer):
    def call(self, x):
        return tf.cast(tf.not_equal(x, 0), tf.float32)

    def get_config(self):
        return super().get_config()

@register_keras_serializable()
class ExpandMaskDim(Layer):
    def call(self, x):
        return tf.expand_dims(x, axis=-1)

    def get_config(self):
        return super().get_config()

class MultiFilterCNNModel:
    """
    Multi-filter CNN model with residual connections for POS tagging (sequence labeling).
    Uses n-gram filters (n = 2, 3, 4) with residual connections.
    """

    def __init__(self, vocab_size, char_vocab_size, embed_dim, output_dim,
                 num_filters=128, pretrained_embeddings=None,
                 num_layers=2, dropout_rate=0.3,
                 max_sequence_length=50, max_word_length=20, char_embed_dim=50):

        self.vocab_size = vocab_size
        self.char_vocab_size = char_vocab_size
        self.embed_dim = embed_dim
        self.output_dim = output_dim
        self.num_filters = num_filters
        self.pretrained_embeddings = pretrained_embeddings
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.max_sequence_length = max_sequence_length
        self.max_word_length = max_word_length
        self.char_embed_dim = char_embed_dim

    def build_character_cnn(self):
        """Build character-level CNN for word embeddings."""
        char_input = Input(shape=(self.max_word_length,), name='char_input')

        # Character embeddings
        char_embed = Embedding(self.char_vocab_size, 32, mask_zero=True)(char_input)
        char_conv = Conv1D(50, 3, activation='relu', padding='same')(char_embed)
        char_pool = GlobalMaxPooling1D()(char_conv)
        char_dense = Dense(self.char_embed_dim, activation='tanh')(char_pool)

        return Model(inputs=char_input, outputs=char_dense, name='char_cnn')

    def build_model(self):
        """Build the complete CNN model for sequence labeling."""
        # Input layers
        word_input = Input(shape=(self.max_sequence_length,), name='word_input')
        char_input = Input(shape=(self.max_sequence_length, self.max_word_length), name='char_input')

        # Word embeddings
        if self.pretrained_embeddings is not None:
            word_embed = Embedding(
                self.vocab_size, self.embed_dim,
                weights=[self.pretrained_embeddings],
                trainable=True, mask_zero=True, name='word_embedding'
            )(word_input)
        else:
            word_embed = Embedding(
                self.vocab_size, self.embed_dim,
                mask_zero=True, name='word_embedding'
            )(word_input)

        # Create mask from word input
        mask = MaskNonPadding()(word_input)
        mask = ExpandMaskDim()(mask)

        # Character-level embeddings
        char_cnn = self.build_character_cnn()
        char_embed = TimeDistributed(char_cnn)(char_input)

        # Combine embeddings
        x = Concatenate()([word_embed, char_embed])
        x = Dense(self.num_filters)(x)
        x = Dropout(self.dropout_rate)(x)

        # Apply mask to embeddings
        x = x * mask

        # Multi-filter CNN with residual connections (2, 3, 4-gram)
        bigram = trigram = fourgram = x

        for i in range(self.num_layers):
            bigram_conv = Conv1D(self.num_filters, 2, padding='same', activation='relu', name=f'bigram_conv_{i}')(bigram)
            bigram = Add(name=f'bigram_residual_{i}')([bigram, bigram_conv])
            bigram = Dropout(self.dropout_rate)(bigram)

            trigram_conv = Conv1D(self.num_filters, 3, padding='same', activation='relu', name=f'trigram_conv_{i}')(trigram)
            trigram = Add(name=f'trigram_residual_{i}')([trigram, trigram_conv])
            trigram = Dropout(self.dropout_rate)(trigram)

            fourgram_conv = Conv1D(self.num_filters, 4, padding='same', activation='relu', name=f'fourgram_conv_{i}')(fourgram)
            fourgram = Add(name=f'fourgram_residual_{i}')([fourgram, fourgram_conv])
            fourgram = Dropout(self.dropout_rate)(fourgram)

        # Concatenate outputs
        multi_filter_concat = Concatenate(name='multi_filter_concat')([bigram, trigram, fourgram])
        multi_filter_concat = Dropout(self.dropout_rate)(multi_filter_concat)

        # Time-distributed classification layer for token-level output
        output = TimeDistributed(
            Dense(self.output_dim, activation='softmax'),
            name='token_classifier'
        )(multi_filter_concat)

        model = Model(inputs=[word_input, char_input], outputs=output, name='multi_filter_cnn_pos_tagger')
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model


In [ ]:
# Function to evaluate predictions for sequence labeling
def evaluate_sequence_predictions(true_tags, pred_tags, tag_to_idx, max_length):
    # Flatten the predictions, excluding padding
    true_flat = []
    pred_flat = []

    for i, (true_seq, pred_seq) in enumerate(zip(true_tags, pred_tags)):
        for j, (true_tag, pred_tag) in enumerate(zip(true_seq, pred_seq)):
            # Skip padding tokens
            if true_tag != 0:  # 0 is padding
                true_flat.append(true_tag)
                pred_flat.append(pred_tag)

    # Calculate metrics
    precision = precision_score(true_flat, pred_flat, average=None, zero_division=0)
    recall = recall_score(true_flat, pred_flat, average=None, zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average=None, zero_division=0)

    macro_precision = precision_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_recall = recall_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_f1 = f1_score(true_flat, pred_flat, average='macro', zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    for i in range(len(tag_to_idx)):
        true_binary = [1 if t == i else 0 for t in true_flat]
        pred_binary = [1 if p == i else 0 for p in pred_flat]

        if sum(true_binary) > 0:  # Only if class exists in true labels
            precision_curve, recall_curve, _ = precision_recall_curve(true_binary, pred_binary)
            pr_auc.append(auc(recall_curve, precision_curve))
        else:
            pr_auc.append(0.0)

    macro_pr_auc = np.mean(pr_auc)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_pr_auc": macro_pr_auc
    }

## 5. Hyperparameter Tuning

Before building our final model, we'll perform hyperparameter tuning to find the optimal values for:
1. CNN hidden size dimension (number of filters)
2. Number of CNN layers
3. Dropout rate

We'll use a grid search approach and evaluate each combination on the development set.


In [ ]:
# # Define hyperparameter grid
# param_grid = {
#     'num_filters': [64, 128, 256],  # CNN hidden size dimension
#     'num_layers': [1, 2, 3],        # Number of CNN layers
#     'dropout_rate': [0.3, 0.5]      # Dropout rate
# }
#
# # Function to train and evaluate a model with specific hyperparameters
# def evaluate_hyperparameters(num_filters, num_layers, dropout_rate, epochs=10):
#     print(f"\nEvaluating model with: filters={num_filters}, layers={num_layers}, dropout={dropout_rate}")
#
#     # Build model with these hyperparameters
#     model = MultiFilterCNNModel(
#         vocab_size=len(word_to_idx),
#         char_vocab_size=len(char_to_idx) if USE_CHAR_EMBEDDINGS else None,
#         embed_dim=EMBEDDING_DIM,
#         output_dim=len(tag_to_idx),
#         num_filters=num_filters,
#         pretrained_embeddings=embedding_matrix,
#         num_layers=num_layers,
#         dropout_rate=dropout_rate,
#         max_sequence_length=MAX_SEQUENCE_LENGTH,
#         max_word_length=MAX_WORD_LENGTH,
#         char_embed_dim=CHAR_EMBEDDING_DIM
#     ).build_model()
#
#     # Early stopping to prevent overfitting during hyperparameter search
#     early_stopping = EarlyStopping(
#         monitor='val_loss',
#         patience=3,
#         restore_best_weights=True,
#         verbose=0
#     )
#
#     # Train the model
#     history = model.fit(
#         train_inputs,
#         y_train,
#         batch_size=BATCH_SIZE,
#         epochs=epochs,
#         validation_data=(dev_inputs, y_dev),
#         callbacks=[early_stopping],
#         verbose=0
#     )
#
#     # Evaluate on dev set
#     dev_pred = model.predict(dev_inputs, verbose=0)
#     dev_pred_classes = np.argmax(dev_pred, axis=-1)
#     dev_metrics = evaluate_sequence_predictions(y_dev, dev_pred_classes, tag_to_idx, MAX_SEQUENCE_LENGTH)
#
#     # Return the evaluation metrics and the number of epochs actually trained
#     return {
#         'num_filters': num_filters,
#         'num_layers': num_layers,
#         'dropout_rate': dropout_rate,
#         'macro_f1': dev_metrics['macro_f1'],
#         'macro_precision': dev_metrics['macro_precision'],
#         'macro_recall': dev_metrics['macro_recall'],
#         'macro_pr_auc': dev_metrics['macro_pr_auc'],
#         'epochs_trained': len(history.history['loss']),
#         'val_loss': min(history.history['val_loss'])
#     }
#
# # Perform grid search
# print("Starting hyperparameter tuning...")
# results = []
#
# for num_filters in param_grid['num_filters']:
#     for num_layers in param_grid['num_layers']:
#         for dropout_rate in param_grid['dropout_rate']:
#             result = evaluate_hyperparameters(num_filters, num_layers, dropout_rate)
#             results.append(result)
#             print(f"  F1: {result['macro_f1']:.4f}, Val Loss: {result['val_loss']:.4f}")
#
# # Convert results to DataFrame for easier analysis
# import pandas as pd
# results_df = pd.DataFrame(results)
# print("\nHyperparameter tuning results:")
# print(results_df.sort_values('macro_f1', ascending=False).head())
#
# # Visualize results
# plt.figure(figsize=(15, 10))
#
# # Plot F1 score by number of filters and layers
# plt.subplot(2, 2, 1)
# for layers in param_grid['num_layers']:
#     layer_results = results_df[results_df['num_layers'] == layers]
#     plt.plot(layer_results['num_filters'], layer_results['macro_f1'],
#              marker='o', label=f'{layers} layers')
# plt.xlabel('Number of Filters')
# plt.ylabel('Macro F1 Score')
# plt.title('F1 Score by Filters and Layers')
# plt.legend()
# plt.grid(True)
#
# # Plot F1 score by dropout rate
# plt.subplot(2, 2, 2)
# dropout_results = results_df.groupby('dropout_rate')['macro_f1'].mean().reset_index()
# plt.bar(dropout_results['dropout_rate'].astype(str), dropout_results['macro_f1'])
# plt.xlabel('Dropout Rate')
# plt.ylabel('Average Macro F1 Score')
# plt.title('F1 Score by Dropout Rate')
# plt.grid(True, axis='y')
#
# # Plot validation loss by number of filters and layers
# plt.subplot(2, 2, 3)
# for layers in param_grid['num_layers']:
#     layer_results = results_df[results_df['num_layers'] == layers]
#     plt.plot(layer_results['num_filters'], layer_results['val_loss'],
#              marker='o', label=f'{layers} layers')
# plt.xlabel('Number of Filters')
# plt.ylabel('Validation Loss')
# plt.title('Validation Loss by Filters and Layers')
# plt.legend()
# plt.grid(True)
#
# # Plot number of epochs trained
# plt.subplot(2, 2, 4)
# epochs_by_params = results_df.groupby(['num_filters', 'num_layers'])['epochs_trained'].mean().reset_index()
# plt.scatter(epochs_by_params['num_filters'], epochs_by_params['num_layers'],
#             s=epochs_by_params['epochs_trained']*20, alpha=0.6)
# plt.xlabel('Number of Filters')
# plt.ylabel('Number of Layers')
# plt.title('Average Epochs Trained')
# for i, row in epochs_by_params.iterrows():
#     plt.annotate(f"{row['epochs_trained']:.1f}",
#                  (row['num_filters'], row['num_layers']),
#                  ha='center', va='center')
# plt.grid(True)
#
# plt.tight_layout()
# plt.show()
#
# # Get the best hyperparameters
# best_result = results_df.loc[results_df['macro_f1'].idxmax()]
# print(f"\nBest hyperparameters:")
# print(f"  Number of filters: {best_result['num_filters']}")
# print(f"  Number of layers: {best_result['num_layers']}")
# print(f"  Dropout rate: {best_result['dropout_rate']}")
# print(f"  Resulting F1 score: {best_result['macro_f1']:.4f}")
#
# # Update the hyperparameters for the final model
# CNN_FILTERS = int(best_result['num_filters'])
# NUM_CNN_LAYERS = int(best_result['num_layers'])
# DROPOUT_RATE = float(best_result['dropout_rate'])


In [ ]:
# Using best instead of tuning
CNN_FILTERS=256
NUM_CNN_LAYERS=3
DROPOUT_RATE=0.3

In [ ]:
# Build CNN model with the best hyperparameters
print(f"Building CNN model with tuned hyperparameters:")
print(f"  - CNN Filters: {CNN_FILTERS}")
print(f"  - Number of CNN Layers: {NUM_CNN_LAYERS}")
print(f"  - Dropout Rate: {DROPOUT_RATE}")

cnn_model = MultiFilterCNNModel(
    vocab_size=len(word_to_idx),
    char_vocab_size=len(char_to_idx) if USE_CHAR_EMBEDDINGS else None,
    embed_dim=EMBEDDING_DIM,
    output_dim=len(tag_to_idx),
    num_filters=CNN_FILTERS,
    pretrained_embeddings=embedding_matrix,
    num_layers=NUM_CNN_LAYERS,
    dropout_rate=DROPOUT_RATE,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_word_length=MAX_WORD_LENGTH,
    char_embed_dim=CHAR_EMBEDDING_DIM
).build_model()

print("\nModel Summary:")
cnn_model.summary()


## 6. Model Training and Evaluation


In [ ]:
# Function to evaluate predictions for sequence labeling
def evaluate_sequence_predictions(true_tags, pred_tags, tag_to_idx, max_length):
    # Flatten the predictions, excluding padding
    true_flat = []
    pred_flat = []

    for i, (true_seq, pred_seq) in enumerate(zip(true_tags, pred_tags)):
        for j, (true_tag, pred_tag) in enumerate(zip(true_seq, pred_seq)):
            # Skip padding tokens
            if true_tag != 0:  # 0 is padding
                true_flat.append(true_tag)
                pred_flat.append(pred_tag)

    # Calculate metrics
    precision = precision_score(true_flat, pred_flat, average=None, zero_division=0)
    recall = recall_score(true_flat, pred_flat, average=None, zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average=None, zero_division=0)

    macro_precision = precision_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_recall = recall_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_f1 = f1_score(true_flat, pred_flat, average='macro', zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    for i in range(len(tag_to_idx)):
        true_binary = [1 if t == i else 0 for t in true_flat]
        pred_binary = [1 if p == i else 0 for p in pred_flat]

        if sum(true_binary) > 0:  # Only if class exists in true labels
            precision_curve, recall_curve, _ = precision_recall_curve(true_binary, pred_binary)
            pr_auc.append(auc(recall_curve, precision_curve))
        else:
            pr_auc.append(0.0)

    macro_pr_auc = np.mean(pr_auc)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_pr_auc": macro_pr_auc
    }

# Set up callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_cnn_pos_tagger.h5', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

# Train the model
print("Training CNN model...")
history = cnn_model.fit(
    train_inputs,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(dev_inputs, y_dev),
    callbacks=callbacks,
    verbose=1
)

# Save best model
cnn_model.save("best_cnn_model.keras")

# Load best model
cnn_model = tf.keras.models.load_model('best_cnn_model.keras')

In [ ]:
# Plot training curves
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curves
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy curves
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# Plot training curves
print("\nPlotting training curves...")
plot_training_curves(history)


In [ ]:
# Make predictions
print("Making predictions...")

# CNN predictions
train_cnn_pred = cnn_model.predict(train_inputs)
dev_cnn_pred = cnn_model.predict(dev_inputs)
test_cnn_pred = cnn_model.predict(test_inputs)

# Convert to class predictions
train_cnn_pred_classes = np.argmax(train_cnn_pred, axis=-1)
dev_cnn_pred_classes = np.argmax(dev_cnn_pred, axis=-1)
test_cnn_pred_classes = np.argmax(test_cnn_pred, axis=-1)

# Evaluate models
print("Evaluating models...")

# Evaluate CNN
train_cnn_metrics = evaluate_sequence_predictions(y_train, train_cnn_pred_classes,
                                                 tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_cnn_metrics = evaluate_sequence_predictions(y_dev, dev_cnn_pred_classes,
                                               tag_to_idx, MAX_SEQUENCE_LENGTH)
test_cnn_metrics = evaluate_sequence_predictions(y_test, test_cnn_pred_classes,
                                               tag_to_idx, MAX_SEQUENCE_LENGTH)

# Evaluate baseline
train_baseline_metrics = evaluate_sequence_predictions(y_train, train_baseline_seq,
                                                      tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_baseline_metrics = evaluate_sequence_predictions(y_dev, dev_baseline_seq,
                                                    tag_to_idx, MAX_SEQUENCE_LENGTH)
test_baseline_metrics = evaluate_sequence_predictions(y_test, test_baseline_seq,
                                                    tag_to_idx, MAX_SEQUENCE_LENGTH)


In [ ]:
# Print results
print("\n" + "="*80)
print("EXPERIMENTAL RESULTS")
print("="*80)

print(f"\nModel Configuration:")
print(f"- CNN Filters: {CNN_FILTERS}")
print(f"- Number of CNN Layers: {NUM_CNN_LAYERS}")
print(f"- Dropout Rate: {DROPOUT_RATE}")
print(f"- Character Embeddings: {USE_CHAR_EMBEDDINGS}")
print(f"- Max Sequence Length: {MAX_SEQUENCE_LENGTH}")
print(f"- Batch Size: {BATCH_SIZE}")

print(f"\nBaseline Model Results:")
print("-" * 40)
print("Training Set:")
print(f"  Macro-averaged Precision: {train_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_baseline_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_baseline_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_baseline_metrics['macro_pr_auc']:.4f}")

print(f"\nCNN Model Results:")
print("-" * 40)
print("Training Set:")
print(f"  Macro-averaged Precision: {train_cnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_cnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_cnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_cnn_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_cnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_cnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_cnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_cnn_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_cnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_cnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_cnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_cnn_metrics['macro_pr_auc']:.4f}")


In [ ]:
print("\nPer-class Metrics for CNN Model (Train Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = train_cnn_metrics['precision'][i]
    recall = train_cnn_metrics['recall'][i]
    f1 = train_cnn_metrics['f1'][i]
    pr_auc = train_cnn_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")

print("\nPer-class Metrics for CNN Model (Test Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = test_cnn_metrics['precision'][i]
    recall = test_cnn_metrics['recall'][i]
    f1 = test_cnn_metrics['f1'][i]
    pr_auc = test_cnn_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")

print("\nPer-class Metrics for CNN Model (Development Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = dev_cnn_metrics['precision'][i]
    recall = dev_cnn_metrics['recall'][i]
    f1 = dev_cnn_metrics['f1'][i]
    pr_auc = dev_cnn_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")


In [ ]:
# Load MLP results from Assignment 3 for comparison
# We just copy pasted the results
mlp_train_metrics = {
    "macro_precision": 0.8789,
    "macro_recall": 0.8636,
    "macro_f1": 0.8702,
    "macro_pr_auc": 0.9023
}

mlp_dev_metrics = {
    "macro_precision": 0.8265,
    "macro_recall": 0.7940,
    "macro_f1": 0.8030,
    "macro_pr_auc": 0.8430
}

mlp_test_metrics = {
    "macro_precision": 0.8335,
    "macro_recall": 0.8091,
    "macro_f1": 0.8144,
    "macro_pr_auc": 0.8540
}

# Load RNN results from Assignment 4 for comparison
rnn_train_metrics = {
    "macro_precision": 0.8827,
    "macro_recall": 0.8070,
    "macro_f1": 0.8246,
    "macro_pr_auc": 0.8460
}

rnn_dev_metrics = {
    "macro_precision": 0.8132,
    "macro_recall": 0.7470,
    "macro_f1": 0.7700,
    "macro_pr_auc": 0.8115
}

rnn_test_metrics = {
    "macro_precision": 0.8294,
    "macro_recall": 0.7727,
    "macro_f1": 0.7948,
    "macro_pr_auc": 0.8323
}

print("\nMLP Model Results (from Assignment 3):\n")
print("Training Set:")
print(f"  Macro-averaged Precision: {mlp_train_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_train_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_train_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_train_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {mlp_dev_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_dev_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_dev_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_dev_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {mlp_test_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_test_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_test_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_test_metrics['macro_pr_auc']:.4f}")

print("\nRNN Model Results (from Assignment 4):\n")
print("Training Set:")
print(f"  Macro-averaged Precision: {rnn_train_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {rnn_train_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {rnn_train_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {rnn_train_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {rnn_dev_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {rnn_dev_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {rnn_dev_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {rnn_dev_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {rnn_test_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {rnn_test_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {rnn_test_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {rnn_test_metrics['macro_pr_auc']:.4f}")


In [ ]:
# Plot the loss curves
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Plot confusion matrix for test set
print("Plotting confusion matrix...")

# Flatten the true and predicted tags for test set
true_tags_flat = []
pred_tags_flat = []
for true_seq, pred_seq in zip(y_test, test_cnn_pred_classes):
    for true_tag, pred_tag in zip(true_seq, pred_seq):
        if true_tag != 0:  # Skip padding
            true_tags_flat.append(true_tag)
            pred_tags_flat.append(pred_tag)

# Create confusion matrix
cm = confusion_matrix(true_tags_flat, pred_tags_flat)

# Normalize the confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))],
            yticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))])
plt.title('Normalized Confusion Matrix')
plt.xlabel('Predicted Tags')
plt.ylabel('True Tags')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Example predictions
print("Example Predictions:")
print("="*60)

def print_example(sentence, true_tags, baseline_pred, cnn_pred, idx_to_tag):
    print("\nSentence:", " ".join(sentence))
    print("\nTrue Tags:", " ".join(true_tags))
    print("Baseline :", " ".join(baseline_pred))
    print("CNN Pred :", " ".join([idx_to_tag[idx] for idx in cnn_pred[:len(sentence)]]))

# Select some examples from test set
example_indices = np.random.choice(len(test_sentences), 5, replace=False)
for i in example_indices:
    sentence = test_sentences[i]
    true_tags = test_pos_tags[i]
    baseline_pred = test_baseline_preds[i]
    cnn_pred = test_cnn_pred_classes[i]
    print_example(sentence, true_tags, baseline_pred, cnn_pred, idx_to_tag)

print("\n" + "="*60)
print("POS Tagging Experiment Completed")
print("="*60)


## 7. Model Comparison and Analysis

In this section, we compare the performance of our CNN model with the baseline, MLP, and RNN models.


In [ ]:
# Create a bar chart to compare model performance
def plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, metric_name):
    models = ['Baseline', 'MLP', 'RNN', 'CNN']
    train_values = [baseline_metrics['train'][metric_name], mlp_metrics['train'][metric_name],
                   rnn_metrics['train'][metric_name], cnn_metrics['train'][metric_name]]
    dev_values = [baseline_metrics['dev'][metric_name], mlp_metrics['dev'][metric_name],
                 rnn_metrics['dev'][metric_name], cnn_metrics['dev'][metric_name]]
    test_values = [baseline_metrics['test'][metric_name], mlp_metrics['test'][metric_name],
                  rnn_metrics['test'][metric_name], cnn_metrics['test'][metric_name]]

    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width, train_values, width, label='Train')
    ax.bar(x, dev_values, width, label='Dev')
    ax.bar(x + width, test_values, width, label='Test')

    ax.set_ylabel(f'{metric_name.replace("macro_", "Macro-averaged ")}')
    ax.set_title(f'Model Comparison - {metric_name.replace("macro_", "Macro-averaged ")}')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.grid(True, axis='y')

    # Add value labels on top of bars
    for i, v in enumerate(train_values):
        ax.text(i - width, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    for i, v in enumerate(dev_values):
        ax.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    for i, v in enumerate(test_values):
        ax.text(i + width, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()

# Organize metrics for comparison
baseline_metrics = {
    'train': train_baseline_metrics,
    'dev': dev_baseline_metrics,
    'test': test_baseline_metrics
}

mlp_metrics = {
    'train': mlp_train_metrics,
    'dev': mlp_dev_metrics,
    'test': mlp_test_metrics
}

rnn_metrics = {
    'train': rnn_train_metrics,
    'dev': rnn_dev_metrics,
    'test': rnn_test_metrics
}

cnn_metrics = {
    'train': train_cnn_metrics,
    'dev': dev_cnn_metrics,
    'test': test_cnn_metrics
}

# Plot comparisons for different metrics
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, 'macro_f1')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, 'macro_precision')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, 'macro_recall')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, 'macro_pr_auc')
